# Phase 6 — Web search tool + escalation

Companion exploration notebook for ROADMAP.md §Phase 6, same shape as
`notebooks/agent_phase5.ipynb`. All the actual logic lives in `rag/websearch.py` (the tool +
the escalation rule) and the small additions to `rag/graph.py`/`rag/generation.py` that wire it
in — this notebook imports them and runs each piece in isolation, rather than re-implementing
anything here. `rag/agent_cli.py --no-web` is the same pipeline as a one-shot CLI; this notebook
is for understanding the mechanism, not a second copy.

No new graph node: escalation is an inline check inside `synthesizer_node`/`single_agent_node`
— both are already the one place each path's final generation call happens, with the fully
merged/reranked chunk set already in hand.

Walk order:
1. `should_escalate()` alone — a well-covered goldset question vs. a documented weak-coverage one
2. `web_search()` alone — one live Tavily call
3. Full graph, well-covered question — no escalation needed
4. Full graph, weakly-covered question — escalation fires, citations shown separately
5. `--no-web` ablation — same question, escalation disabled, reverts to the original abstention
6. Threshold tuning — the measured numbers behind `config.WEB_ESCALATION_THRESHOLD`

**Needs a running Qdrant index** (`ai`/`cs.CL`/`finance`, per PHASE3_NOTES.md), `GOOGLE_API_KEY`
and `TAVILY_API_KEY` in `.env`. Embedded-mode Qdrant is single-writer — close `retriever` (last
cell) before running `rag.cli`/`rag.agent_cli`/`rag.web_eval` from a separate process.

In [1]:
# Unlike the Phase 1 notebooks (which predate rag/ and are fully self-contained), this one
# imports from the rag package - Jupyter's cwd is this notebook's own directory, not the repo
# root, so `from rag import ...` needs the root on sys.path first.
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "rag" / "__init__.py").exists():
    if _root.parent == _root:
        raise RuntimeError("could not find the project root (looked for rag/__init__.py)")
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [2]:
from rag import config
from rag.generation import Generator
from rag.graph import build_graph, run_agent
from rag.retrieval import Retriever
from rag.websearch import should_escalate, web_search

D:\.tutorials\agentic-rag-capstone\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Constructed once, reused for the whole notebook - matches rag/eval.py's run_tier23 style.
# Model/index loads happen lazily on first use (rag/retrieval.py, rag/generation.py), so this
# cell itself is instant; the first retrieval/generation call below is the slow one.
retriever = Retriever()
generator = Generator()

## 1. `should_escalate()` alone

`rag/websearch.py`'s escalation rule is one comparison: `chunks[0].score < config.WEB_ESCALATION_THRESHOLD`.
`chunks[0].score` is meaningful because `Reranker.rerank()` overwrites `.score` with the
cross-encoder score - checking it after a `use_rerank=True` search genuinely checks the
*reranker's* top score. One question from the curated goldset (guaranteed well-covered, since
`rag/goldset.py` generated it from a real indexed chunk), one that PHASE5B_NOTES.md §5 already
documented as a corpus-coverage gap (both turns of that conversation abstained) - real weak
retrieval, not a contrived one.

In [4]:
print("WEB_ESCALATION_THRESHOLD =", config.WEB_ESCALATION_THRESHOLD)

# From goldset_curated.jsonl (finance category) - answerable from the corpus by construction.
strong_chunks = retriever.search(
    "What is the Settlement Modernisation Index (SMI) designed to capture?",
    use_bm25=True, use_rerank=True,
)
print(f"well-covered question | top score={strong_chunks[0].score:.4f} | "
      f"should_escalate={should_escalate(strong_chunks)}")

WEB_ESCALATION_THRESHOLD = 0.48619261384010315


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 36109.23it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7255.43it/s]

well-covered question | top score=0.9972 | should_escalate=False


In [5]:
# PHASE5B_NOTES.md §5's own worked example: this exact question ("its assumptions?" follow-up
# included) abstained end-to-end - the 20-docs/category finance corpus (PHASE3_NOTES.md) simply
# doesn't cover Black-Scholes well, a documented coverage gap, not a made-up weak case.
weak_chunks = retriever.search(
    "What is the Black-Scholes model used for in options pricing?", use_bm25=True, use_rerank=True,
)
print(f"weakly-covered question | top score={weak_chunks[0].score:.4f} | "
      f"should_escalate={should_escalate(weak_chunks)}")

weakly-covered question | top score=0.0048 | should_escalate=True


## 2. `web_search()` alone

One Tavily call via `langchain-tavily`, results taken as-is - no multi-query, no reranking, no
credibility scoring (ROADMAP's explicit cut list). Fails open to `[]` on a missing key or API
error, same posture as the planner/router/contextualize LLM calls.

In [6]:
results = web_search("What is the Black-Scholes model used for in options pricing?")
for r in results:
    print(f"- {r.title}\n  {r.url}\n  {r.snippet[:160]}...\n")

- Black-Scholes Model: What It Is, How It Works, and the Options Formula
  https://www.investopedia.com/terms/b/blackscholes.asp
  ## Frequently Asked Questions

## What Does the Black-Scholes Model Do?

The Black-Scholes model, also known as the Black-Scholes-Merton (BSM), was the first wi...

- What is the Black-Scholes Equation? | Black-Scholes Model Explained | CQF
  https://www.cqf.com/blog/quant-finance-101/what-is-the-black-scholes-equation
  The Black-Scholes equation is a mathematical formula used to price European-style options, which are options that can only be exercised at expiration. It was de...

- Black-Scholes Model (BSM) & Understanding the Value of a Stock Option
  https://carta.com/learn/startups/equity-management/black-scholes-model
  The Black-Scholes option pricing model (also called the Black and Scholes Model, the Black-Scholes-Merton Model, or simply “BSM”) is one of the most commonly us...



## 3. Full graph, well-covered question - no escalation needed

`run_agent(..., use_web=True)` (the default) only calls `web_search()` when `should_escalate()`
fires inside `synthesizer_node`/`single_agent_node`. The SMI question from step 1 should come
back with `final_answer.web_results` empty and a fully corpus-cited answer.

In [7]:
import uuid

graph = build_graph(retriever, generator)

in_corpus_result = run_agent(
    "What is the Settlement Modernisation Index (SMI) designed to capture?", retriever, generator,
    use_fanout=False, thread_id=f"notebook-phase6-incorpus-{uuid.uuid4()}", graph=graph,
)
answer = in_corpus_result["final_answer"]
print("web_results:", len(answer.web_results), "(expect 0 - no escalation needed)")
print()
print(answer.text)

web_results: 0 (expect 0 - no escalation needed)

The Settlement Modernisation Index (SMI) is a longitudinal measure designed to capture the evolution of wholesale cross-border settlement infrastructures and their associated balance-sheet efficiencies across advanced economies since 1993 [1]. It specifically measures the regulatory, technological, and market-practice completion of reform events [3]. By doing so, the index provides a framework to capture the dimensions that determine the effectiveness of wholesale settlement infrastructures over time [1].


## 4. Full graph, weakly-covered question - escalation fires

This is the row-6 arm ("+ web fallback", ROADMAP §4) closing a real gap: PHASE5B_NOTES.md §5
documented this exact question abstaining end-to-end because the corpus doesn't cover it well.
Now the reranker's top score falls below `WEB_ESCALATION_THRESHOLD`, `single_agent_node` calls
`web_search()`, and the final answer can cite corpus sources as `[1]`, `[2]` and web sources as
`[W1]`, `[W2]` - two separate numbering spaces, never merged (PHASE6_NOTES.md §3).

In [8]:
escalated_result = run_agent(
    "What is the Black-Scholes model used for in options pricing?", retriever, generator,
    use_fanout=False, thread_id=f"notebook-phase6-escalate-{uuid.uuid4()}", graph=graph,
)
escalated_answer = escalated_result["final_answer"]
print("web_results:", len(escalated_answer.web_results), "(expect > 0 - escalation should fire)")
print()
print(escalated_answer.text)
print()
print("corpus cited:", escalated_answer.cited, "| web cited:", escalated_answer.web_cited)

web_results: 3 (expect > 0 - escalation should fire)

The Black-Scholes model is a mathematical formula used to calculate the fair price or theoretical value of European-style options [W1][W2]. It determines this value by accounting for variables such as the underlying asset's current price, the strike price, the time until expiration, the risk-free interest rate, and the asset's volatility [W1][W2]. Additionally, the model is recognized as a standard method for valuing stock options, as it considers both the intrinsic value and the time value of an option [W3].

corpus cited: [] | web cited: [1, 2, 3]


In [9]:
if escalated_answer.sources():
    print("Corpus sources")
    for handle, chunk in escalated_answer.sources():
        print(f"  [{handle}] {chunk.provenance()}")

if escalated_answer.web_sources():
    print("\nWeb sources (separate from the corpus above)")
    for handle, result in escalated_answer.web_sources():
        print(f"  [W{handle}] {result.title}  {result.url}")


Web sources (separate from the corpus above)
  [W1] Black-Scholes Model: What It Is, How It Works, and the Options Formula  https://www.investopedia.com/terms/b/blackscholes.asp
  [W2] What is the Black-Scholes Equation? | Black-Scholes Model Explained | CQF  https://www.cqf.com/blog/quant-finance-101/what-is-the-black-scholes-equation
  [W3] Black-Scholes Model (BSM) & Understanding the Value of a Stock Option  https://carta.com/learn/startups/equity-management/black-scholes-model


## 5. `--no-web` ablation

Same question, `use_web=False`. `should_escalate()` never even runs - the ablation flag is a
zero-cost bypass, same pattern as `use_planner`/`use_fanout`/`use_context`. Without a web
fallback, this reverts to exactly what PHASE5B_NOTES.md §5 originally documented: the corpus
alone can't answer it, so it abstains.

In [10]:
no_web_result = run_agent(
    "What is the Black-Scholes model used for in options pricing?", retriever, generator,
    use_fanout=False, use_web=False,
    thread_id=f"notebook-phase6-noweb-{uuid.uuid4()}", graph=graph,
)
no_web_answer = no_web_result["final_answer"]
print("web_results:", len(no_web_answer.web_results), "(expect 0 - --no-web disables escalation)")
print("abstained:", no_web_answer.abstained)
print()
print(no_web_answer.text)

web_results: 0 (expect 0 - --no-web disables escalation)
abstained: True

INSUFFICIENT_CONTEXT
The provided sources do not contain information regarding the use of the Black-Scholes model in options pricing.


## 6. Threshold tuning

`config.WEB_ESCALATION_THRESHOLD` isn't guessed - `rag/web_eval.py` measures the reranked top-1
score `Retriever.search()` returns on every one of `goldset_curated.jsonl`'s questions (all
answerable by construction) and sets the threshold at the 5th percentile of that distribution.
See PHASE6_NOTES.md §2 for why the golden set can bound the false-escalation rate but not the
catch rate on genuinely unanswerable questions yet.

In [11]:
import json

report = json.loads(config.WEB_ESCALATION_REPORT_PATH.read_text())
print(json.dumps(report["summary"], indent=2))

{
  "n": 123,
  "percentile": 0.05,
  "threshold": 0.48619261384010315,
  "min": 0.07782477140426636,
  "max": 0.9999308586120605,
  "mean": 0.9220582750148889,
  "median": 0.9900396466255188,
  "would_escalate_at_threshold": 6,
  "false_escalation_rate": 0.04878048780487805
}


---

For the one-shot CLI form, see `rag/agent_cli.py` (`--no-web` to disable escalation), or
`rag/web_eval.py` to re-tune `WEB_ESCALATION_THRESHOLD` after the corpus or goldset changes.
Design decisions and defense notes are in `PHASE6_NOTES.md`.

Run the next cell before starting `rag.cli`/`rag.agent_cli`/`rag.web_eval` from a terminal -
embedded-mode Qdrant is single-writer, and this notebook's `retriever` is still holding the
storage lock.

In [12]:
retriever.close()